# 프롬프트 체이닝(Prompt Chaining) 실습

프롬프트 체이닝은 복잡한 작업을 여러 단계로 나누어 LLM에게 요청하는 기술입니다. 각 단계에서 생성된 출력을 다음 단계의 입력으로 사용하는 방식으로 체인을 형성합니다.

## 1. 프롬프트 체이닝 소개

프롬프트 체이닝은 다음과 같은 이점이 있습니다:

- **복잡한 작업 분해**: 하나의 복잡한 작업을 여러 작은 작업으로 나누어 처리합니다.
- **성능 향상**: LLM이 한번에 복잡한 작업을 처리하는 것보다 단계별로 처리할 때 더 나은 결과를 얻을 수 있습니다.
- **투명성과 제어**: 각 단계별로 결과를 확인하고 디버깅할 수 있어 문제 해결이 용이합니다.
- **개인화와 사용자 경험 향상**: 단계별 응답을 통해 더 맞춤화된 결과를 제공할 수 있습니다.

In [15]:
# 유틸리티 함수 가져오기
import sys
import os

# 프로젝트 루트 디렉토리를 파이썬 경로에 추가
sys.path.append(os.path.abspath('../'))

from utils.helpers import (
    get_completion,
    get_chat_completion,
    create_few_shot_prompt
)

## 2. 기본 프롬프트 체이닝 예제

먼저 간단한 예제를 통해 프롬프트 체이닝의 기본 개념을 이해해봅시다. 이 예제에서는 두 단계로 구성된 프롬프트 체인을 만들 것입니다:
1. 첫 번째 단계: 주어진 주제에 대한 아이디어 생성
2. 두 번째 단계: 생성된 아이디어를 바탕으로 블로그 글 작성

In [16]:
def simple_prompt_chain(topic):
    # 첫 번째 프롬프트: 아이디어 생성
    ideas_prompt = f"다음 주제에 대한 5가지 흥미로운 아이디어를 생성해주세요: {topic}"
    
    # API 호출
    ideas = get_completion(ideas_prompt, temperature=0.7)
    print("[1단계 결과] 생성된 아이디어:\n", ideas)
    
    # 두 번째 프롬프트: 블로그 글 작성
    blog_prompt = f"다음 아이디어를 바탕으로 짧은 블로그 글의 도입부를 작성해주세요:\n{ideas}"
    
    # API 호출
    blog_post = get_completion(blog_prompt, temperature=0.7)
    
    return blog_post

In [17]:
# 프롬프트 체인 실행하기
topic = "인공지능의 미래"
result = simple_prompt_chain(topic)
print("\n[최종 결과] 생성된 블로그 글:\n")
print(result)

[1단계 결과] 생성된 아이디어:
 인공지능의 미래에 대한 흥미로운 아이디어는 다양하고 혁신적일 수 있습니다. 다음은 그 중 다섯 가지입니다:

1. **AI와 인간의 협업 강화**: 미래에는 AI와 인간이 협력하여 문제를 해결하는 방식이 더욱 발전할 것입니다. 예를 들어, AI는 의료 진단에서 방대한 데이터를 분석하여 의사에게 최적의 치료 옵션을 제안할 수 있으며, 이는 치료의 정확성과 효율성을 크게 향상시킬 것입니다.

2. **AI 윤리 및 규제 발전**: 인공지능의 발전과 함께 윤리적 문제와 규제가 더욱 중요해질 것입니다. AI가 인간의 삶에 미치는 영향을 최소화하고 긍정적인 방향으로 발전시키기 위해, AI 윤리 기준과 국제 규제가 더욱 발전하고 표준화될 것입니다.

3. **개인 맞춤형 AI 비서**: AI가 개인의 생활 패턴, 선호도, 목표를 학습하여 완전히 개인화된 비서 역할을 수행할 것입니다. 이러한 AI 비서는 개인의 일정 관리부터 건강 관리, 금융 계획까지 다양한 영역에서 개인화된 조언을 제공할 수 있습니다.

4. **자연어 처리의 혁신**: 자연어 처리 기술이 더욱 발전하여 AI가 인간의 복잡한 감정, 뉘앙스, 맥락을 이해하고 대화할 수 있는 능력이 향상될 것입니다. 이는 고객 서비스, 교육, 심리 상담 등 다양한 분야에서 활용될 수 있습니다.

5. **AI와 창의성의 융합**: 인공지능이 예술, 디자인, 음악 등 창의적인 분야에서 인간과 협업하여 새로운 형태의 예술을 창조할 것입니다. AI는 창의적인 프로세스를 지원하고, 인간 창작자와 함께 새로운 아이디어와 스타일을 탐구하는 데 기여할 수 있습니다.

[최종 결과] 생성된 블로그 글:

인공지능 기술이 빠르게 발전하면서, 우리는 AI가 우리의 삶을 어떻게 변화시킬지 고민하게 됩니다. AI의 미래는 단순히 기술 발전에 그치지 않고, 인간과의 협업을 통해 새로운 가능성을 열어가는 방향으로 나아가고 있습니다. 이번 블로그 글에서는 AI의 미래에 대한 다섯 가지 흥미로운 아이디어를 살펴보겠습

## 3. 문서 QA를 위한 프롬프트 체이닝

이제 더 실용적인 예제로 넘어가 봅시다. 문서에서 질문에 답변하는 시스템을 만들어 보겠습니다. 이 시스템은 두 단계로 구성됩니다:
1. 문서에서 질문과 관련된 인용구 추출
2. 추출된 인용구를 바탕으로 질문에 답변 작성

In [18]:
# 샘플 문서 생성
sample_document = """
프롬프트 엔지니어링(Prompt Engineering)은 대규모 언어 모델(LLM)이 특정 작업을 효과적으로 수행하도록 최적화된 프롬프트를 설계하는 과정입니다.

프롬프트 엔지니어링 기법에는 다음과 같은 종류가 있습니다:
1. 제로샷 프롬프팅(Zero-shot Prompting): 예시 없이 바로 작업을 지시합니다.
2. 퓨샷 프롬프팅(Few-shot Prompting): 몇 가지 예시를 제공하여 모델이 패턴을 학습하도록 합니다.
3. 체인오브소트(Chain-of-Thought): 모델이 단계적으로 생각하도록 유도합니다.
4. 자기 일관성(Self-Consistency): 여러 경로를 통해 추론한 후 가장 일관된 답변을 선택합니다.
5. 리액트(ReAct): 추론과 행동을 번갈아가며 수행하는 방식입니다.
6. 검색 증강 생성(RAG): 외부 지식 소스에서 검색한 정보를 결합하여 응답을 생성합니다.
7. 프롬프트 체이닝(Prompt Chaining): 여러 단계의 프롬프트를 연결하여 복잡한 작업을 수행합니다.

효과적인 프롬프트를 작성하려면 명확한 지시사항, 충분한 맥락 제공, 단계별 안내, 다양한 예시 제공 등의 방법을 활용할 수 있습니다.
"""

In [19]:
def document_qa_prompt_chain(document, question):
    # 1단계: 관련 인용구 추출
    extraction_prompt = f"""
    당신은 문서에서 질문과 관련된 인용구를 추출하는 도우미입니다. 
    아래 문서에서 다음 질문과 관련된 인용구를 추출해주세요. 
    인용구는 <quotes></quotes> 태그로 감싸서 출력해주세요.
    
    문서:
    {document}
    
    질문: {question}
    """
    
    # 첫 단계 API 호출 (온도 낮게 설정하여 정확도 높이기)
    quotes = get_completion(extraction_prompt, temperature=0.1)
    print("[1단계 결과] 추출된 인용구:\n", quotes)
    
    # 2단계: 답변 작성
    answer_prompt = f"""
    당신은 질문에 대답하는 도우미입니다.
    다음 문서와 추출된 인용구를 바탕으로 질문에 대한 답변을 작성해주세요.
    답변은 정확하고 친절한 톤으로 작성해주세요.
    
    문서:
    {document}
    
    추출된 인용구:
    {quotes}
    
    질문: {question}
    """
    
    # 두 번째 단계 API 호출
    answer = get_completion(answer_prompt, temperature=0.3)
    
    return answer

In [20]:
# 문서 QA 체인 실행하기
question = "프롬프트 엔지니어링의 다양한 기법에는 어떤 것들이 있나요?"
result = document_qa_prompt_chain(sample_document, question)
print("\n[최종 결과] 질문에 대한 답변:\n")
print(result)

[1단계 결과] 추출된 인용구:
 <quotes>프롬프트 엔지니어링 기법에는 다음과 같은 종류가 있습니다:
1. 제로샷 프롬프팅(Zero-shot Prompting): 예시 없이 바로 작업을 지시합니다.
2. 퓨샷 프롬프팅(Few-shot Prompting): 몇 가지 예시를 제공하여 모델이 패턴을 학습하도록 합니다.
3. 체인오브소트(Chain-of-Thought): 모델이 단계적으로 생각하도록 유도합니다.
4. 자기 일관성(Self-Consistency): 여러 경로를 통해 추론한 후 가장 일관된 답변을 선택합니다.
5. 리액트(ReAct): 추론과 행동을 번갈아가며 수행하는 방식입니다.
6. 검색 증강 생성(RAG): 외부 지식 소스에서 검색한 정보를 결합하여 응답을 생성합니다.
7. 프롬프트 체이닝(Prompt Chaining): 여러 단계의 프롬프트를 연결하여 복잡한 작업을 수행합니다.</quotes>

[최종 결과] 질문에 대한 답변:

프롬프트 엔지니어링에는 다양한 기법이 있으며, 각각의 기법은 대규모 언어 모델이 특정 작업을 효과적으로 수행하도록 돕습니다. 다음은 주요 기법들입니다:

1. **제로샷 프롬프팅(Zero-shot Prompting)**: 예시 없이 바로 작업을 지시하는 방법입니다.
2. **퓨샷 프롬프팅(Few-shot Prompting)**: 몇 가지 예시를 제공하여 모델이 패턴을 학습하도록 하는 방법입니다.
3. **체인오브소트(Chain-of-Thought)**: 모델이 단계적으로 생각하도록 유도하는 기법입니다.
4. **자기 일관성(Self-Consistency)**: 여러 경로를 통해 추론한 후 가장 일관된 답변을 선택하는 방식입니다.
5. **리액트(ReAct)**: 추론과 행동을 번갈아가며 수행하는 방식입니다.
6. **검색 증강 생성(RAG)**: 외부 지식 소스에서 검색한 정보를 결합하여 응답을 생성하는 기법입니다.
7. **프롬프트 체이닝(Prompt Chaining)**: 여러 단계의 프롬프트를 연결하여 복잡한 

## 4. 코드 생성을 위한 프롬프트 체이닝

코드 생성은 프롬프트 체이닝이 매우 유용한 또 다른 영역입니다. 다음 예제에서는 세 단계의 체인을 만들어 보겠습니다:
1. 코드 요구사항 분석 및 설계
2. 실제 코드 작성
3. 코드 테스트 및 디버깅

In [21]:
def code_generation_prompt_chain(task_description):
    # 1단계: 코드 요구사항 분석 및 설계
    design_prompt = f"""
    당신은 소프트웨어 설계자입니다. 다음 작업에 대한 요구사항을 분석하고, 
    필요한 기능과 구조를 설계해주세요.
    
    작업 설명: {task_description}
    
    다음 형식으로 응답해주세요:
    1. 기능 요구사항
    2. 필요한 함수/클래스
    3. 데이터 구조
    4. 알고리즘 설계
    """
    
    # 채팅 형식으로 시스템 프롬프트 설정
    design_messages = [
        {"role": "system", "content": "당신은 전문적인 소프트웨어 설계자입니다."},
        {"role": "user", "content": design_prompt}
    ]
    
    # 첫 단계 API 호출
    design = get_chat_completion(design_messages, temperature=0.2)
    print("[1단계 결과] 코드 설계:\n", design)
    
    # 2단계: 코드 작성
    coding_prompt = f"""
    당신은 프로그래머입니다. 다음 설계 문서를 기반으로 코드를 작성해주세요.
    
    작업 설명: {task_description}
    
    설계 문서:
    {design}
    
    위 설계에 따라 Python 코드를 작성해주세요. 주석을 포함하고 모든 함수를 구현해주세요.
    """
    
    # 채팅 형식으로 시스템 프롬프트 설정
    coding_messages = [
        {"role": "system", "content": "당신은 전문적인 Python 프로그래머입니다."},
        {"role": "user", "content": coding_prompt}
    ]
    
    # 두 번째 단계 API 호출
    code = get_chat_completion(coding_messages, temperature=0.1)
    print("\n[2단계 결과] 생성된 코드:\n", code)
    
    # 3단계: 코드 테스트 및 디버깅
    testing_prompt = f"""
    당신은 QA 엔지니어입니다. 다음 코드를 테스트하고 잠재적인 문제와 개선사항을 식별해주세요.
    
    작업 설명: {task_description}
    
    코드:
    {code}
    
    다음 사항을 포함해주세요:
    1. 코드의 잠재적 버그 또는 오류
    2. 성능 개선 가능성
    3. 최종 개선된 코드
    """
    
    # 채팅 형식으로 시스템 프롬프트 설정
    testing_messages = [
        {"role": "system", "content": "당신은 전문적인 QA 엔지니어입니다."},
        {"role": "user", "content": testing_prompt}
    ]
    
    # 세 번째 단계 API 호출
    improved_code = get_chat_completion(testing_messages, temperature=0.2)
    
    return improved_code

In [22]:
# 코드 생성 체인 실행하기
task = "사용자로부터 온도를 입력받아 섭씨에서 화씨로, 또는 화씨에서 섭씨로 변환하는 프로그램을 작성해주세요."
result = code_generation_prompt_chain(task)
print("\n[최종 결과] 테스트 및 개선된 코드:\n")
print(result)

[1단계 결과] 코드 설계:
 1. 기능 요구사항
   - 사용자는 온도를 입력할 수 있어야 합니다.
   - 사용자는 변환할 온도의 단위를 선택할 수 있어야 합니다 (섭씨에서 화씨 또는 화씨에서 섭씨).
   - 프로그램은 입력된 온도를 선택된 단위로 변환하여 출력해야 합니다.
   - 사용자가 잘못된 입력을 했을 경우, 적절한 오류 메시지를 출력하고 재입력을 요청해야 합니다.

2. 필요한 함수/클래스
   - `TemperatureConverter`: 온도 변환을 담당하는 클래스.
     - `convert_to_fahrenheit(celsius: float) -> float`: 섭씨를 화씨로 변환하는 메서드.
     - `convert_to_celsius(fahrenheit: float) -> float`: 화씨를 섭씨로 변환하는 메서드.
   - `main()`: 프로그램의 진입점으로 사용자 입력을 처리하고 변환 결과를 출력하는 함수.
   - `get_user_input() -> Tuple[float, str]`: 사용자로부터 온도와 변환 방향을 입력받는 함수.
   - `validate_input(temperature: str, unit: str) -> bool`: 입력된 온도와 단위가 유효한지 검증하는 함수.

3. 데이터 구조
   - 사용자 입력을 저장하기 위한 변수:
     - `temperature`: 변환할 온도의 값 (float).
     - `unit`: 변환할 온도의 단위 ('C' 또는 'F').

4. 알고리즘 설계
   - `main()` 함수:
     1. `get_user_input()` 함수를 호출하여 사용자로부터 온도와 변환 방향을 입력받습니다.
     2. 입력된 값이 유효한지 `validate_input()` 함수를 통해 검증합니다.
     3. 입력이 유효하지 않으면 오류 메시지를 출력하고 재입력을 요청합니다.
     4. 입력이 유효하면, `TemperatureConverter` 클래스의 적절한 메서

## 5. 복잡한 추론을 위한 프롬프트 체이닝

이제 더 복잡한 추론 과정을 여러 단계로 나누어 수행하는 예제를 살펴보겠습니다. 여기서는 수학 문제 풀이를 단계별로 진행하겠습니다.

In [23]:
def math_problem_solving_chain(problem):
    # Few-shot 예제 준비
    examples = [
        {
            "input": "마트에서 사과 3개를 샀는데 각각 1200원이었다. 총 얼마를 지불해야 하는가?",
            "output": "사과 한 개의 가격은 1200원이고, 사과를 3개 샀으므로 총 금액은 1200원 × 3 = 3600원이다. 따라서 총 3600원을 지불해야 한다."
        },
        {
            "input": "A 기차는 시속 80km로 달리고, B 기차는 시속 120km로 달린다. 두 기차가 서로 반대 방향으로 출발하여 3시간 후에 만났다면, 처음 두 기차 사이의 거리는 얼마인가?",
            "output": "A 기차는 시속 80km로 3시간 동안 이동했으므로 80km × 3 = 240km를 이동했다. B 기차는 시속 120km로 3시간 동안 이동했으므로 120km × 3 = 360km를 이동했다. 두 기차가 만났을 때 이동한 총 거리가 처음 두 기차 사이의 거리이므로, 240km + 360km = 600km이다. 따라서 처음 두 기차 사이의 거리는 600km이다."
        }
    ]
    
    # 1단계: 문제 분석
    analysis_instruction = "다음 수학 문제를 분석하고 어떤 개념이 필요한지, 어떤 접근 방식을 사용해야 하는지 설명해주세요."
    
    # Few-shot 프롬프트 생성
    analysis_prompt = create_few_shot_prompt(
        instruction=analysis_instruction,
        examples=examples,
        query=problem
    )
    
    # 첫 단계 API 호출
    analysis = get_completion(analysis_prompt, temperature=0.2)
    print("[1단계 결과] 문제 분석:\n", analysis)
    
    # 2단계: 단계별 풀이
    solving_prompt = f"""
    당신은 수학 교사입니다. 다음 수학 문제를 분석 결과를 바탕으로 단계별로 풀이해주세요.
    각 단계를 명확하게 설명하고, 사용하는 공식이나 개념을 언급해주세요.
    
    문제: {problem}
    
    문제 분석:
    {analysis}
    """
    
    # 채팅 형식으로 시스템 프롬프트 설정
    solving_messages = [
        {"role": "system", "content": "당신은 인내심 있는 수학 교사입니다."},
        {"role": "user", "content": solving_prompt}
    ]
    
    # 두 번째 단계 API 호출
    solution = get_chat_completion(solving_messages, temperature=0.3)
    print("\n[2단계 결과] 단계별 풀이:\n", solution)
    
    # 3단계: 정답 확인 및 요약
    verification_prompt = f"""
    당신은 수학 검증자입니다. 다음 수학 문제의 풀이를 검증하고 최종 답을 간결하게 요약해주세요.
    또한 풀이에 오류가 있는지 확인하고, 있다면 수정해주세요.
    
    문제: {problem}
    
    풀이:
    {solution}
    """
    
    # 세 번째 단계 API 호출
    verification = get_completion(verification_prompt, temperature=0.1, max_tokens=500)
    
    return verification

In [24]:
# 수학 문제 풀이 체인 실행하기
math_problem = "한 농장에 닭과 토끼가 있습니다. 머리는 총 35개이고 다리는 총 94개입니다. 닭과 토끼는 각각 몇 마리인가요?"
result = math_problem_solving_chain(math_problem)
print("\n[최종 결과] 검증 및 요약:\n")
print(result)

[1단계 결과] 문제 분석:
 이 문제는 닭과 토끼의 수를 구하는 문제로, 연립 방정식을 사용하여 해결할 수 있습니다. 문제를 분석하고 필요한 개념과 접근 방식을 설명하겠습니다.

### 문제 분석
1. **주어진 정보:**
   - 머리의 총 개수는 35개입니다.
   - 다리의 총 개수는 94개입니다.
   - 닭은 머리 1개와 다리 2개를 가지고 있습니다.
   - 토끼는 머리 1개와 다리 4개를 가지고 있습니다.

2. **목표:**
   - 닭과 토끼의 수를 각각 구합니다.

### 필요한 개념
- **연립 방정식:** 두 개의 미지수를 포함하는 두 개의 방정식을 사용하여 문제를 해결합니다.
- **대수적 조작:** 방정식을 변형하고 조작하여 해를 구합니다.

### 접근 방식
1. **변수 설정:**
   - \( x \)를 닭의 수, \( y \)를 토끼의 수라고 합시다.

2. **방정식 구성:**
   - 머리의 수에 대한 방정식: \( x + y = 35 \)
   - 다리의 수에 대한 방정식: \( 2x + 4y = 94 \)

3. **방정식 풀이:**
   - 첫 번째 방정식에서 \( x = 35 - y \)로 표현할 수 있습니다.
   - 이 표현을 두 번째 방정식에 대입합니다: 
     \[
     2(35 - y) + 4y = 94
     \]
   - 이를 풀면:
     \[
     70 - 2y + 4y = 94
     \]
     \[
     2y = 94 - 70
     \]
     \[
     2y = 24
     \]
     \[
     y = 12
     \]
   - \( y = 12 \)를 첫 번째 방정식에 대입하여 \( x \)를 구합니다:
     \[
     x + 12 = 35
     \]
     \[
     x = 23
     \]

### 최종 답변
닭은 23마리이고, 토끼는 12마리입니다.

[2단계 결과] 단계별 풀이:
 ### 단계별 풀이

1. **문제 이해 및 

## 6. 대화형 프롬프트 체이닝

대화 맥락을 유지하면서 여러 단계의 상호작용을 수행하는 예제를 살펴보겠습니다.

In [25]:
def conversational_prompt_chain():
    # 대화 맥락 초기화
    conversation = [
        {"role": "system", "content": "당신은 여행 계획을 도와주는 친절한 AI 도우미입니다. 사용자의 선호도를 파악하고 맞춤형 여행 계획을 제안해주세요."}
    ]
    
    # 첫 번째 질문: 선호도 파악
    conversation.append({"role": "user", "content": "한국에서 3박 4일 여행을 계획 중입니다. 도움을 주실 수 있나요?"})
    response1 = get_chat_completion(conversation, temperature=0.7)
    print("[AI 응답 1]:\n", response1)
    
    # AI 응답 추가
    conversation.append({"role": "assistant", "content": response1})
    
    # 두 번째 질문: 구체적인 정보 제공
    conversation.append({"role": "user", "content": "저는 역사적인 장소와 자연 경관을 좋아합니다. 그리고 한식을 즐기고 싶어요. 서울과 부산을 방문할 계획입니다."})
    response2 = get_chat_completion(conversation, temperature=0.7)
    print("\n[AI 응답 2]:\n", response2)
    
    # AI 응답 추가
    conversation.append({"role": "assistant", "content": response2})
    
    # 세 번째 질문: 결과 종합
    conversation.append({"role": "user", "content": "좋은 제안 감사합니다. 이제 제가 선택한 일정을 바탕으로 3박 4일 상세 여행 계획을 만들어주세요."})
    response3 = get_chat_completion(conversation, temperature=0.7, max_tokens=1000)
    
    return response3

In [26]:
# 대화형 체인 실행하기
result = conversational_prompt_chain()
print("\n[최종 결과] 상세 여행 계획:\n")
print(result)

[AI 응답 1]:
 물론이죠! 한국에서의 3박 4일 여행을 멋지게 계획해드리겠습니다. 몇 가지 질문을 드릴게요. 어떤 도시를 방문하고 싶으신가요? 예를 들어 서울, 부산, 제주도 등이 있습니다. 그리고 특별히 관심 있는 활동이나 음식이 있으신가요? 문화 유적지 탐방, 자연 경관 감상, 쇼핑, 미식 탐방 등 여러 옵션이 있습니다. 여행 스타일에 따라 맞춤형 추천을 드릴 수 있어요.

[AI 응답 2]:
 좋습니다! 서울과 부산에서 역사적인 장소와 자연 경관을 감상하고 한식을 즐기는 3박 4일 여행 일정을 제안드리겠습니다.

### 첫째 날: 서울
- **경복궁**: 조선 시대의 주요 궁궐 중 하나로, 한국의 역사와 문화를 느낄 수 있습니다. 오전에 방문하여 수문장 교대식도 관람하세요.
- **북촌 한옥마을**: 전통 한옥이 모여 있는 곳으로, 한국의 옛 주거 문화를 경험할 수 있습니다. 골목길을 걸으며 사진 찍기 좋습니다.
- **인사동 거리**: 다양한 전통 공예품과 기념품을 구입할 수 있는 곳입니다. 한식당과 전통 찻집도 많이 있습니다.

### 둘째 날: 서울
- **남산 서울타워**: 서울의 전경을 한눈에 볼 수 있는 명소입니다. 케이블카를 타고 올라가거나, 산책로를 따라 걸어 올라갈 수 있습니다.
- **남대문 시장**: 다양한 먹거리를 즐길 수 있는 전통 시장입니다. 길거리 음식에서부터 다양한 한식까지 맛볼 수 있습니다.
- **청계천**: 도심 속에 위치한 아름다운 산책로입니다. 저녁에는 조명이 켜져 낭만적인 분위기를 즐길 수 있습니다.

### 셋째 날: 부산
- **해운대 해수욕장**: 부산의 대표적인 해변으로, 아름다운 해변 경관을 감상하며 여유로운 시간을 보낼 수 있습니다.
- **광안리 해수욕장**: 광안대교가 보이는 멋진 풍경을 자랑하는 또 다른 해변입니다. 저녁에는 해변가의 카페에서 휴식을 취하세요.
- **부산 국제시장**: 부산의 대표적인 전통시장으로, 다양한 먹거리와 쇼핑을 즐길 수 있습니다.

### 넷째 날: 부산
- **범어

## 7. 요약

이 노트북에서는 프롬프트 체이닝의 다양한 예제를 살펴보았습니다:

1. **기본 프롬프트 체이닝**: 아이디어 생성 후 블로그 글 작성
2. **문서 QA 체이닝**: 문서에서 인용구 추출 후 질문 답변
3. **코드 생성 체이닝**: 설계, 코딩, 테스트의 3단계 체인
4. **수학 문제 풀이 체이닝**: 분석, 풀이, 검증의 3단계 체인
5. **대화형 프롬프트 체이닝**: 맥락을 유지하며 단계적으로 정보 수집 및 결과 생성

프롬프트 체이닝은 다양한 복잡한 작업을 LLM으로 해결하는 데 매우 유용한 기법입니다. 각 단계가 명확하게 분리되어 있어 디버깅이 용이하고, 각 단계에서 최적의 결과를 얻을 수 있도록 프롬프트를 조정할 수 있습니다.

## 8. 연습 문제

1. 다음 작업을 3단계 이상의 프롬프트 체인으로 구현해보세요:
   - 뉴스 기사 요약 및 감정 분석
   - 소설 또는 시나리오 생성
   - 데이터 분석 및 시각화 계획 수립

2. 여러 단계의 프롬프트를 사용하되, 각 단계의 출력을 다음 단계의 입력으로 사용하는 방식이 아닌 병렬 처리 방식의 프롬프트 체이닝을 구현해보세요.

3. 사용자의 피드백을 반영하여 결과를 개선하는 반복적인 프롬프트 체인을 설계해보세요.